In [ ]:
!pip install vnstock -U
from vnstock import Vnstock
from google.colab import drive
from vnstock.api.quote import Quote
import pandas as pd
from datetime import datetime

# Kết nối với Google Drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.4/277.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 3.9 MB/s eta 0:00:00
Mounted at /content/drive


Crawl test

In [ ]:
# Crawl data mã VCI qua KBS
quote = Quote(symbol='VCI', source='KBS')
df = quote.history(start='2024-01-01', end='2024-05-25', interval="1D")

# In 5 dòng đầu tiên của dữ liệu
print(df.head())

# Lưu dữ liệu ra file Excel
df.to_excel('gia_lich_su_ACB.xlsx', index=False)
print("Đã lưu dữ liệu vào file 'gia_lich_su_ACB.xlsx'")

                 time   open   high    low  close   volume
0 2024-01-02 07:00:00  23.61  23.61  23.08  23.22  7174900
1 2024-01-03 07:00:00  23.17  23.44  22.95  23.44  3331800
2 2024-01-04 07:00:00  23.50  23.96  23.30  23.30  9108900
3 2024-01-05 07:00:00  23.39  23.44  23.14  23.33  3824400
4 2024-01-08 07:00:00  23.47  23.77  23.41  23.44  4107100
Đã lưu dữ liệu vào file 'gia_lich_su_ACB.xlsx'


Crawl data của HĐTL gần nhất & hiện tại

In [ ]:
def collect_derivative_data():
    print("--- Bắt đầu crawl data ---")

    START_DATE = '2020-01-01'
    END_DATE = datetime.now().strftime('%Y-%m-%d')

    # Crawl data VN30F1M
    print(f"1. Đang lấy dữ liệu lịch sử VN30F1M từ {START_DATE}...")
    q_f1m = Quote(symbol='VN30F1M', source='KBS')
    df_f1m = q_f1m.history(start=START_DATE, end=END_DATE, interval="1h")

    # Crawl data VN30
    print("2. Đang lấy dữ liệu chỉ số cơ sở VN30...")
    q_vn30 = Quote(symbol='VN30', source='KBS')
    df_vn30 = q_vn30.history(start=START_DATE, end=END_DATE, interval="1h")

    # Đồng bộ & merge dữ liệu
    df_f1m = df_f1m[['time', 'open', 'high', 'low', 'close', 'volume']].rename(columns={'close': 'y_f1m'})
    df_vn30 = df_vn30[['time', 'close']].rename(columns={'close': 'vn30_index'})
    raw_data = pd.merge(df_f1m, df_vn30, on='time', how='inner')

    # Xuất dữ liệu ra CSV
    file_path = '/content/drive/MyDrive/vn30f1m_raw_dataset.csv'
    raw_data.to_csv(file_path, index=False, encoding='utf-8-sig')

    print(f"--- Hoàn tất! ---")
    print(f"Số dòng: {len(raw_data)}")
    print(f"Lưu tại: {file_path}")
    print(raw_data.head())

Thử crawl data của HĐTL đã đáo hạn

In [ ]:
def collect_historical_f1m():
    # Danh sách các mã hợp đồng trong quá khứ
    symbols = ['VN30F2512', 'VN30F2601', 'VN30F2602', 'VN30F2603', 'VN30F2604']
    all_data = []

    for sym in symbols:
        print(f"Đang lấy dữ liệu mã: {sym}")
        try:
            q = Quote(symbol=sym, source='KBS')
            # Lấy toàn bộ lịch sử của mã đó
            df = q.history(start='2025-01-01', end='2026-05-11', interval="1D")
            if not df.empty:
                all_data.append(df)
        except:
            print(f"Mã {sym} không lấy được dữ liệu.")

    # Merge data
    if all_data:
        full_f1m = pd.concat(all_data).drop_duplicates(subset='time').sort_values('time')
        # Lưu file
        full_f1m.to_csv('/content/drive/MyDrive/vn30f1m_deep_history.csv', index=False)
        print("--- Hoàn tất! Đã nối dữ liệu các tháng thành file duy nhất ---")
        print(f"Tổng số dòng: {len(full_f1m)}")
    else:
        print("Không lấy được dữ liệu nào.")

Thực thi

In [ ]:
print("--- Đang lấy dữ liệu từ hợp đồng tương lai gần nhất ---")
collect_derivative_data()

print("--- Đang lấy dữ liệu từ hợp đồng tương lai trước đó ---")
collect_historical_f1m()

--- Đang lấy dữ liệu từ hợp đồng tương lai gần nhất ---
--- Bắt đầu crawl data ---
1. Đang lấy dữ liệu lịch sử VN30F1M từ 2020-01-01...
2. Đang lấy dữ liệu chỉ số cơ sở VN30...
--- Hoàn tất! ---
Số dòng: 168
Lưu tại: /content/drive/MyDrive/vn30f1m_raw_dataset.csv
                 time    open    high     low   y_f1m  volume  vn30_index
0 2026-03-20 09:00:00  1846.0  1846.0  1814.6  1821.3     212     1829.20
1 2026-03-20 10:00:00  1822.9  1826.6  1811.3  1811.6     168     1825.24
2 2026-03-20 11:00:00  1815.5  1815.5  1813.5  1814.8       4     1821.15
3 2026-03-20 13:00:00  1814.5  1822.4  1808.6  1808.9     127     1811.88
4 2026-03-20 14:00:00  1807.5  1810.6  1794.0  1799.8     193     1797.99
--- Đang lấy dữ liệu từ hợp đồng tương lai trước đó ---
Đang lấy dữ liệu mã: VN30F2512
Mã VN30F2512 không lấy được dữ liệu.
Đang lấy dữ liệu mã: VN30F2601
Mã VN30F2601 không lấy được dữ liệu.
Đang lấy dữ liệu mã: VN30F2602

📋 Kết nối tài khoản Google Drive để lưu các thiết lập của dự án.
Dữ 